# DWthon Adaptive Planning: Dzień 1

## Dlaczego pętla jest potrzebna?

Witam Cię na pokładzie ostatniego DWthonu. Nazywam się **[Vladimir Alekseichenko](https://www.linkedin.com/in/vladimiralekseichenko/)** (zaproś mnie do swojej siecie, już ponad 25k ludzi to zrobili). W IT działam od 2008 roku, w Machine Learningu zawodowo od ponad dekady, a z LLM-ami pracuję od premiery Transformerów (między innymi od czasów, kiedy OpenAI publikowało otwarcie wagi swoich modeli). Najbardziej interesuje mnie nie pytanie: "który model jest najlepszy?". Ważniejsze jest pytanie: **jak zbudować system, który działa, gdy człowiek nie patrzy mu na ręce?**

Przez trzy dni zbudujemy kawałek mojego konceptu "Mój Robot". Dostanie proste pytanie o technologię, sam zaplanuje badanie, po każdym wyniku sprawdzi, czy plan nadal ma sens, a na końcu odda raport Robota: rekomendację, dowody, założenia i listę rzeczy, których nie wie.

Dzień 1 jest o zrozumieniu mechanizmu. Nie będziemy dziś budować dużego systemu. Najpierw chcę, żebyś zobaczył, dlaczego zwykły agent z narzędziami nie wystarcza do długiego zadania.

### Dwa błędy, które widzę najczęściej

Pierwszy: ktoś projektuje całą ścieżkę z góry, a potem dziwi się, że agent ślepo idzie dalej mimo nowych danych.

Drugi: ktoś mówi modelowi "bądź mądry" i daje mu pełną swobodę. Wtedy otrzymuje efektowne odpowiedzi, ale nie potrafi powiedzieć, skąd wzięła się decyzja ani czego system nie zrobi w złym scenariuszu.

Poziom 5 jest pomiędzy tymi skrajnościami. Robot ma swobodę drogi, ale nie ma swobody zmiany celu, fałszowania dowodów ani ignorowania budżetu.


In [ ]:
from dotenv import load_dotenv
load_dotenv()

import importlib
import day1_runtime
importlib.reload(day1_runtime)

from day1_runtime import (
    load_day1_artifacts, show_autonomy_levels, show_contract, show_experiment,
    show_fixed_report, show_no_change, show_plan_patch, show_report_preview,
    show_snapshot, show_trace,
)

snapshot, replay = load_day1_artifacts()
print(f"✓ Wczytano snapshot z {snapshot['captured_at']} i prawdziwy replay modelu.")
show_autonomy_levels()

## Czym poziom 5 nie jest?

Nie jest to system, który sam ustala sobie (pierwotne/nadrzędne) cele. Nie jest to też system, który ma odpowiedź na każde pytanie. Poziom 5 oznacza konkretną kompetencję: po każdym wyniku potrafi zdecydować, co warto sprawdzić dalej. To jest bliższe pracy dobrego analityka niż pracy chatbota. Dajesz mu temat, kryteria oraz budżet. Nie piszesz za niego listy wszystkich kroków. Oczekujesz natomiast, że pokaże Ci drogę, dowody i moment, w którym musi wrócić do Ciebie z pytaniem.


## Kontrakt zamiast długiego promptu

W moim koncepcie nie zaczyna od wielostronicowego formularza. Użytkownik zadaje proste pytanie: "Czy warto zrobić eksperyment dotyczący AI w code review?". Model interpretuje je jako kontrakt: decyzja, horyzont, standard dowodu, budżet i frazy wyszukiwania.

To ważne. Dlaczego? Plan może się zmieniać wielokrotnie. Takie życie, rzeka płynie... Kontrakt stoi, dopóki człowiek go jawnie nie zmieni. Dzięki temu robot może dojść tam, gdzie nie byliśmy na początku, ale nie może po cichu zmienić problemu na łatwiejszy. Swoją drogą, jak bardzo mózg człowieka lubi optymalizować zużycie energii - to, co po ludzku nazywamy "lenieniem się", LLM-y też mają w pewnym sensie wbudowane. Ale to już, jak się mówi: "z kim przestajesz, takim się stajesz" 😄.

Zobacz przykładowy kontrakt, nic nadzwyczajnego. Zawsze możesz go rozbudować lub zmienić. Do naszego przykładu to wystarczy.

In [ ]:
show_contract(replay["contract"])

### Plan nie jest kontraktem

To rozróżnienie jest całym fundamentem. Warto złapać tę ideę, bo między innymi narzuca ona ramy dla AI/LLM. Kontrakt mówi: "pomóż mi zdecydować, czy warto zrobić eksperyment w ciągu 90 dni". Plan pyta: "jakich sygnałów szukamy, jaki dowód jest potrzebny i co może zmniejszyć ryzyko tej decyzji?". Jeśli po drodze okaże się, że katalog narzędzi nie przybliża nas do decyzji, robot może usunąć takie pytanie. Jeśli odkryje, że największy problem to szum w źródłach, może dodać pytanie o zakres zmian kierunku. To nie jest zmiana celu. To jest dobra praca nad celem.

Ta różnica jest bardzo praktyczna. Użytkownik nie musi umieć zaprojektować researchu. Wystarczy, że naturalnie powie: "Pomóż zdecydować, czy w ciągu 90 dni warto zrobić eksperyment AI aby robić code review. Nie rekomenduj działania bez źródła pierwotnego i kontrargumentu. Jeśli brakuje kontekstu biznesowego, zapytaj mnie". Robot zamienia to na kontrakt, a nie na tysiąc kroków.


## Skąd robot bierze wiedzę?

W tym DWthonie celowo zawężamy świat do Hacker News. HN jest radarem technicznych sygnałów: są tam nowe wpisy, dyskusje, linki do repozytoriów, dokumentacji i case studies. Nie jest źródłem prawdy. Dlatego robot rozdziela sygnał od dowodu. Wątek może powiedzieć nam, że problem jest ważny. Link do README albo opisu wdrożenia może dać coś znacznie mocniejszego. Jeżeli źródła nie da się zweryfikować, raport powie to wprost.

In [ ]:
show_snapshot(snapshot)

## Dlaczego nie używamy tu pełnego Deep Research?

Na to jest dedykowany cały moduł (tydzień czasu) w kursie [Agentic AI](https://dataworkshop.eu/agentic-ai) gdzie używasz Tavily, wielu domen i większej liczby wymiarów badania. To jest następny krok. Tutaj chcemy najpierw zobaczyć wzorzec bez ukrycia go za złożonością. Jedno źródło pozwala wyraźnie zobaczyć, kiedy plan się zmienia. Ten sam rdzeń później dostanie więcej źródeł, pamięć i harmonogram. Najpierw jednak musimy umieć zbudować pętlę, której ufamy.

## Zasada całego DWthonu

Nie uczysz się tu trzech osobnych trików. Patrzysz na jeden system z trzech odległości. Dzisiaj zobaczysz jego decyzje i nazwiesz wzorzec. Jutro otworzymy środek: stan, plan, `NextStep`, Pydantic i dispatcher. Trzeciego dnia ten sam silnik dostanie ekran Gradio, a Ty sprawdzisz jego granice własnymi kliknięciami.

### Po dniu 1 powinieneś umieć odpowiedzieć na 7 pytań

1. Dlaczego sam dobry prompt nie naprawia sztywnej pętli?
2. Co odróżnia poziom 4 od poziomu 5 autonomii?
3. Dlaczego plan jest hipotezą, a nie checklistą?
4. Co człowiek ustala raz, a co model może zmieniać po drodze?
5. Co dokładnie zapisuje `NextStep`?
6. Jak `Union` ogranicza decyzje modelu bez odbierania mu sprawczości?
7. Jak rozpoznać, że robot adaptuje plan, a nie tylko tworzy nową odpowiedź?


## DataWorkshop: pragmatyzm ponad hype

W DataWorkshop nie interesuje nas budowanie kolejnego efektownego demo, które działa tylko wtedy, gdy ktoś dokładnie pilnuje promptu. Interesują nas systemy, za które można wziąć odpowiedzialność.

Dlatego oddzielamy trzy rzeczy, które rynek bardzo często miesza:

1. **Model**: potrafi rozumieć i generować język.
2. **Narzędzie**: potrafi coś pobrać, zapisać albo sprawdzić.
3. **System**: ma cel, stan, granice i sposób reagowania na to, co wydarzyło się po drodze.

Nowy model, framework agentowy albo kolejne narzędzie nie tworzą automatycznie autonomii. To są narzędzia. Odpowiedzialna autonomia pojawia się dopiero wtedy, gdy zaprojektujesz sposób podejmowania kolejnych decyzji. To ważne także biznesowo. Firma nie kupuje modelu po to, żeby model ładnie napisał odpowiedź. Kupuje krótszy czas dojścia do dobrej decyzji, mniej ręcznego researchu i mniej błędów zrobionych dlatego, że ktoś realizował stary plan mimo nowych informacji. Jeśli nie umiemy opisać, co system robi po wyniku narzędzia, nie mamy jeszcze agenta. Mamy czat z dodatkami.

## Software 1.0, 2.0 i 3.0

Pewnie to już wiesz (zawsze warto powtórzyć), to ważne rozróżnienie, bo inaczej bardzo łatwo pomylić "model odpowiada" z "system działa".

### Software 1.0: reguły w kodzie

Programista zapisuje wszystkie kroki sam. To świetne podejście, gdy świat jest przewidywalny i reguły są znane.

### Software 2.0: reguły w danych

Model uczy się wzorca z wielu przykładów. Dobrze radzi sobie ze statystyką, ale trudno zmienić pojedynczą decyzję albo zrozumieć, dlaczego tak zadziałał.

### Software 3.0: część logiki jest zapisana w języku

LLM potrafi czytać cel, kontekst i wynik działania narzędzia. Dzięki temu może podjąć decyzję, której programista nie umiał zapisać z góry jako prostego `if`. Ale właśnie dlatego potrzebuje systemu wokół siebie.

> Software 3.0 nie oznacza: "oddajmy wszystko modelowi". Oznacza: "dajmy modelowi decyzje, których kod nie umie przewidzieć, i otoczmy je kontrolą".

Weź prosty przykład. Kod bez problemu porówna datę wpisu z horyzontem 90 dni. Kod nie wie jednak sam, czy po pięciu opiniach i jednym README nadal badamy to samo pytanie, czy trzeba zmienić pytanie z "czy warto" na "w jakim zakresie warto". To jest decyzja oparta na znaczeniu kilku obserwacji. Właśnie w tym miejscu wykorzystujemy LLM.

## 7 Poziomów autonomii AI

Słowo "agent" niewiele dziś znaczy. Chat z długim promptem też ktoś może nazwać agentem. Agent z narzędziami również. Dlatego potrzebujemy mapy, która mówi **jakie decyzje system naprawdę podejmuje sam**.

Dla tego DWthonu najważniejsze są dwa poziomy:

| Poziom 4: kontekst i narzędzia | Poziom 5: Adaptive Planning |
| --- | --- |
| Agent ma pamięć i potrafi wykonać krok narzędziem. | Robot po wyniku ocenia, czy dotychczasowy plan nadal prowadzi do celu. |
| Może wybrać narzędzie w ramach otrzymanego kroku. | Może dodać pytanie, porzucić nieistotne pytanie albo zmienić priorytet. |
| Przypomina juniora z dobrą checklistą. | Przypomina samodzielnego analityka z celem, budżetem i momentem eskalacji. |

Test jest prosty: czy dałbyś tej osobie temat w piątek, nie pytał o kolejne kroki, a w środę oczekiwał konkretnej rekomendacji albo dobrze postawionego pytania? Jeśli tak, mówimy o kompetencji poziomu 5. To nie znaczy, że ta osoba może robić wszystko. Samodzielny analityk nie zmienia sobie briefu, nie fałszuje cytatów, nie wydaje nieograniczonego budżetu i nie podpisuje decyzji za firmę. Ma za to prawo sam wybrać kolejność pytań, odrzucić ślepą uliczkę oraz wrócić z konkretnym pytaniem, gdy odpowiedź człowieka rzeczywiście zmieni dalszą pracę.

![](../images/7_levels.jpg)


## Paradoks kontroli

Często słyszę: "Chcę autonomicznego agenta, żeby nie musieć go pilnować". To błąd logiczny.

> **Im wyższa autonomia, tym wyższa musi być kontrola systemowa.**

Pomyśl, jak byłoby inaczej. Co to oznaczałoby dla nas, ludzi? Nie chodzi o to, żeby człowiek zatwierdzał każdy klik. To nie skaluje się i nie jest autonomią. Chodzi o kontrolę zaprojektowaną wcześniej:
- człowiek określa cel, granice i budżet
- model wybiera drogę po kolejnych obserwacjach
- kod sprawdza fakty, dozwolone akcje i warunki zakończenia
- każda decyzja zostaje zapisana z powodem

Nie kontrolujemy trasy. Kontrolujemy pojazd. To jest cały temat tego DWthonu. Zły projekt próbuje kontrolować wszystko naraz: każdą decyzję wpisuje w kod i nazywa to autonomią. Drugi zły projekt daje modelowi wolną rękę i dopiero po błędzie próbuje ustalić, co się wydarzyło. My robimy inaczej. Człowiek ustala granicę. Model podejmuje kolejne decyzje. Dispatcher sprawdza, czy wolno je wykonać. Historia zachowuje dowód drogi.


## Co zbudujemy w 3 dni

### Dzień 1: rozumiem
Zobaczysz, dlaczego pętla jest potrzebna i poznasz język, którym będziemy opisywać decyzje robota.

### Dzień 2: potrafię
Otworzymy rdzeń: kontrakt, stan, pytania badawcze z ID, `NextStep`, `PlanPatch`, dispatcher i dowody.

### Dzień 3: czuję wartość
Uruchomisz gotową aplikację Gradio. Zmienisz własne założenie, zobaczysz replan, świadomą nie-zmianę planu i moment, w którym robot pyta człowieka. Na końcu dostaniesz Raport Robota.

To nie będą trzy różne rzeczy. To będzie ten sam robot oglądany z trzech odległości. Na koniec lepiej zrozumiesz ideę i zobaczysz namacalny wynik (tak jak zwykle).

## Dlaczego długie zadania są inne

Jedno pytanie w czacie można zamknąć jedną odpowiedzią. Długie zadanie wygląda inaczej. Pierwszy krok nie daje odpowiedzi. Daje nową informację, która może zmienić sens kroku drugiego. Wyobraź sobie research o AI code review. Na początku pytasz: "czy warto?". Po pierwszym wątku widzisz, że problemem nie jest dostępność narzędzi, tylko szum (swoją drogą, obecnie na temat AI jest bardzo dużo szumu. Trzeba umieć go filtrować, bo inaczej ciężko zrozumieć, co jest najważniejsze i co ma sens). Po lekturze wdrożenia odkrywasz, że wartość zależy od klasy zmian i jakości testów. Dobra odpowiedź nie brzmi już "tak" albo "nie". Brzmi: "zbadaj wąsko, na tej klasie zmian, z tą miarą sukcesu".

Nikt nie mógł uczciwie napisać tej drogi w kroku zero.

To jest różnica między pipeline'em a zadaniem. Pipeline (z góre określone kroki) jest świetny, gdy wiemy, że krok 1 zawsze prowadzi do kroku 2. Długie zadanie zaczyna się tam, gdzie wynik kroku 1 może powiedzieć: "krok 2 nie ma już sensu" albo "brakuje pytania, którego nie było na starcie". Plan w takim świecie nie jest instrukcją. Jest roboczą hipotezą.


## Deklarujesz cel, nie projektujesz trasy

Podam Ci przykład ze światu danych. To przypomina SQL. W SQL mówisz, jakiego wyniku potrzebujesz. Nie rozpisujesz ręcznie, jak silnik ma przejść po indeksach i tabelach. System dobiera drogę do celu, ale działa według jasno określonych reguł. *Mój Robot* dostaje kontrakt: decyzję, horyzont, standard dowodu i budżet. To jest deklaracja. Plan jest już czymś innym. Jest roboczą hipotezą, którą można zmieniać, gdy dane pokazują lepszą drogę.

Ta analogia ma granicę: baza danych jest deterministyczna, LLM nie. Dlatego w naszym systemie potrzebujemy dodatkowo typów, historii decyzji i dispatchera.

## Jakie decyzje podejmuje robot?

Na poziomie 5 LLM nie decyduje, co jest celem ani co jest prawdą. Decyduje, jak dojść do celu. W praktyce oznacza to, że po drodze może:

1. wybrać najważniejsze pytanie badawcze;
2. zdecydować, co oznacza nowa obserwacja;
3. dodać pytanie, którego nie było w pierwszym planie;
4. porzucić pytanie, które przestało przybliżać do decyzji;
5. zmienić priorytet pytań;
6. zdecydować, kiedy potrzebuje człowieka;
7. zdecydować, czy ma już podstawy, aby skończyć.

To jest autonomia, którą da się poczuć. Nie dlatego, że robot robi wszystko. Dlatego, że sam podejmuje wiele sensownych decyzji i żadnej nie ukrywa.


## Czego robot nie może zdecydować

Są rzeczy, których nie oddajemy modelowi. Człowiek ustala cel i może zmienić kontrakt. Kod sprawdza, czy cytat naprawdę występuje w źródle, czy data mieści się w horyzoncie i czy decyzja nie przekracza budżetu. Robot nie może też sam powiedzieć "działaj", jeżeli nie ma wymaganego dowodu i kontrargumentu. Może wybrać rekomendację tylko spośród tych, na które pozwala aktualny stan. To nie ogranicza inteligencji. To sprawia, że inteligencja ma gdzie bezpiecznie pracować.

## Test Adaptive Planning

Po tym DWthonie możesz ocenić każdy system, który ktoś nazwie autonomicznym (na poziomie 5). Zadaj trzy pytania:

1. Czy końcowy plan dało się uczciwie napisać przed pierwszą obserwacją? Jeśli tak, to prawdopodobnie pipeline.
2. Czy robot potrafił zmienić plan także wtedy, gdy kod nie zgłosił prostego błędu? Jeśli nie, to prawdopodobnie system reguł.
3. Czy robot potrafił świadomie nie zmienić planu po kuszącym, ale nieistotnym sygnale? Jeśli nie, reaguje nerwowo, a nie adaptacyjnie.

W dniu 3 przejdziesz wszystkie trzy testy samodzielnie w Gradio.

## Co będzie efektem końcowym

Na końcu nie dostaniesz zwykłego podsumowania wątku. Dostaniesz Raport Robota: rekomendację, dowody, kontrargumenty, drogę decyzji, założenia oraz sekcję "Czego nie wiem". Ta ostatnia część jest bardzo ważna. Dobry system nie udaje, że zebrał wszystko. Mówi, czego nie sprawdził, dlaczego i czy to może zmienić decyzję. Dla biznesu to często bardziej wartościowe niż sama rekomendacja.


In [ ]:
show_report_preview(replay)

## Eksperyment: dlaczego pętla jest potrzebna?

Teraz przejdźmy do prostego pytania: "Co zmieniło się ostatnio wokół AI code review?".

System sztywny może mieć bardzo dobry prompt i poprawnie wykonać: `search -> top 3 -> podsumowanie`. Problem zaczyna się po otrzymaniu wyników. Jeśli dwa najpopularniejsze wpisy są stare, a pytanie dotyczy tego, co wydarzyło się ostatnio, jego plan właśnie przestał pasować do celu. W tej komórce nie ma atrapowego `if`. Wczytujemy zamrożony wynik HN i trace nagrany wcześniej na prawdziwym modelu. Dzięki temu wszyscy widzą ten sam fakt ze świata oraz tę samą decyzję `NextStep`.


In [ ]:
show_experiment(snapshot, replay)

### Co ma zobaczyć uczestnik?

Raport systemu sztywnego może nawet wyglądać przekonująco, ale pierwsze pozycje będą stare. Nie będziemy mówić, że to porażka. Data przy wyniku zrobi to sama. Robot dostanie tę samą obserwację. Różnica jest taka, że po niej dostanie jeszcze jedną turę. W tej turze może powiedzieć: "popularność nie odpowiada na pytanie o świeżość, zmieniam strategię".

In [ ]:
show_fixed_report(snapshot)

## Adaptive Planning: definicja

Adaptive Planning nie jest generowaniem nowego planu. To kontrolowana zmiana planu z zachowaniem tożsamości kroków, kontraktu i historii decyzji. Zatrzymaj się na moment i przeczytaj uważnie, bo to jest ważne. Słowo "kontrolowana" jest tutaj kluczowe. Możemy później sprawdzić, który krok został dodany, który porzucony, co robot zobaczył i dlaczego wykonał kolejną akcję. To odróżnia robota od czatu, który po prostu generuje nowy tekst.

## Dwa wzorce pod maską

### `NextStep`: decyzja, nie kolejna odpowiedź

To nie jest odpowiedź dla użytkownika. To mała, typowana notatka robota po każdej turze: co zobaczył, co jest teraz najważniejsze, czy plan wymaga poprawki i jaka jedna akcja ma wydarzyć się dalej. Najważniejsza jest kolejność. Robot najpierw widzi wynik narzędzia. Dopiero potem podejmuje kolejną decyzję. Dzięki temu nie udaje, że zna drogę przed rozpoczęciem pracy.

### Schema-Guided Reasoning: swoboda wewnątrz ram

Model nie może nagle wymyślić szóstej, nieznanej akcji. W danej turze dostaje tylko decyzje dopuszczone przez kod:

```text
search_hn | read_thread | fetch_source | ask_human | finish
```

Gdy system widzi, że kolejne wyszukiwania nic nie wnoszą, `search_hn` znika z tej ramy. Model nie jest wtedy proszony, żeby był rozsądny. Po prostu nie ma takiej opcji. To nazywamy `Schema-Guided Reasoning`.

To nie jest kaganiec dla modelu. To podział odpowiedzialności. Model decyduje, która z dozwolonych dróg ma teraz sens. Kod gwarantuje, że decyzja da się bezpiecznie wykonać, zmierzyć i później odtworzyć.

```text
stan: są nowe dowody                 stan: 3 puste wyszukiwania

Union[Search, Read, Fetch, Ask, Finish]   Union[Read, Fetch, Ask, Finish]
                 model wybiera                         search znika
```

Zwróć uwagę na subtelną różnicę. Kod nie mówi modelowi: "teraz przeczytaj wątek". Kod mówi tylko: "nie możesz już szukać". To zostawia robotowi realną decyzję i jednocześnie usuwa drogę, która przestała być rozsądna.

### `NextStep`: jedna decyzja na turę

`NextStep` nie jest kolejną odpowiedzią dla użytkownika. To notatka samodzielnego analityka: co właśnie zobaczyłem, które pytanie jest teraz najważniejsze, czy muszę zmienić plan i co robię dalej. Dzięki temu po siedmiu turach mamy drogę, a nie tylko końcowy tekst. Możemy wrócić do dowolnego momentu i sprawdzić decyzję podjętą wtedy, a nie wymyśloną po fakcie. Poniżej masz całą pętlę. Nie musisz dziś implementować każdego pola. Chcę tylko, żebyś zobaczył, gdzie jest miejsce na myślenie modelu, a gdzie zaczyna się odpowiedzialność kodu.


```text

NEXT STEP (odpowiedź LLM)

  co zobaczyłem        situation_analysis
          |
          v
  co jest ważne        current_priority
          |
          v
  czy zmieniam plan?   plan_patch | None
          |
          v
  co robię teraz?      next_action: Union[...]

LLM -> NextStep -> dispatcher() -> obserwacja -> LLM
```

To są trzy wzorce w jednym małym obiekcie. `situation_analysis` daje modelowi ciągłość myślenia. `plan_patch` tworzy cykl, bo plan może zostać poprawiony po wyniku. `next_action` jest routerem, bo wybiera jedną akcję z `Union`. Jutro zobaczysz dokładnie ten obiekt w Pydantic.

### Patch, a nie nowy plan jako tekst

Plan ma pytania ze stabilnymi ID. Robot może dodać pytanie, zamknąć je, porzucić albo zmienić priorytet. Nie przepisuje całej listy, bo wtedy nie wiedzielibyśmy, co faktycznie się zmieniło.

To będzie szczególnie ważne jutro. Zobaczysz diff `Plan v1 -> Plan v2`, a nie nowy akapit udający plan.

In [ ]:
show_plan_patch(replay)

### Nie każda informacja wymaga replanningu

Dojrzały robot nie reaguje nerwowo na wszystko. Głośny wpis może mieć setki punktów, ale być poza horyzontem albo poza kontraktem. Wtedy właściwa decyzja brzmi: `plan_patch=None`. To nie jest brak działania. To jest jawna decyzja: widzę informację, rozumiem ją i wiem, dlaczego nie zmienia mojego celu.


In [ ]:
show_no_change(replay)

### Kod i model mają inne kompetencje

Model interpretuje znaczenie kilku faktów razem. Kod nie powinien udawać, że umie to przewidzieć. Kod robi coś innego: sprawdza cytat, datę, budżet, strukturę patcha i listę dozwolonych działań.

Kod nigdy nie pisze patcha za model. Jeżeli to zrobimy, dostaniemy system reguł z dekoracją LLM, a nie Adaptive Planning.

### Transparentność daje zaufanie

Autonomia nie ma oznaczać czarnej skrzynki. Na osi decyzji zobaczysz: co robot dostał, co widział, co postanowił i dlaczego. Jeśli kod odrzuci decyzję, robot zobaczy powód i podejmie następną próbę. To jest różnica między delegowaniem pracy analitykowi a puszczaniem procesu samopas.


In [ ]:
show_trace(replay)

## Jutro przechodzimy od idei do silnika

Dzień 2 nie będzie próbą napisania wszystkiego od zera. Dostaniesz działający rdzeń, a jeśli wybierzesz ścieżkę "Buduję", uzupełnisz trzy miejsca, które są naprawdę ważne.

Zobaczysz też trudniejszy przypadek: kod zobaczy opinię, repozytorium i case wdrożeniowy. Każdy fakt osobno będzie poprawny. Dopiero model zrozumie, że nie można wrzucić ich do jednego worka.


In [ ]:
assert any(turn.get("plan_patch") for turn in replay["turns"]), "Replay nie zawiera PlanPatch."
assert any(turn.get("plan_patch") is None for turn in replay["turns"]), "Replay nie zawiera plan_patch=None."

print("✓ Day 1 jest gotowy: trace zawiera zmianę planu i świadomą nie-zmianę.")

## Zadanie 1.1: zaprojektuj granice Mojego Robota

Nie budujesz jeszcze całej aplikacji. Robisz ważniejszą rzecz: sprawdzasz, czy umiesz oddzielić cel od drogi. Wybierz jeden temat technologiczny, który naprawdę pojawia się w Twojej pracy. Nie musi to być AI code review. Może to być local-first, nowy framework albo narzędzie, które ktoś właśnie chce wdrożyć w Twoim zespole.

Uzupełnij trzy zdania w kolejnej komórce. Nie pisz planu krok po kroku. Napisz tylko: jaką decyzję ma wesprzeć robot, co uznasz za wystarczający dowód i kiedy robot powinien wrócić do Ciebie z pytaniem. To jest kontrakt. Dopiero po nim robot ma prawo wybrać drogę.

### Podpowiedź

<details>
<summary><b>👉 Kliknij, jeśli potrzebujesz podpowiedzi</b></summary>

Dobra decyzja nie brzmi "sprawdź wszystko o AI". Musi dać się podjąć po raporcie robota.

- Decyzja: "Czy warto zrobić 2-tygodniowy spike AI code review w naszym zespole?"
- Dowód: "Nie rekomenduj działania bez jednego przykładu użycia i jednego ograniczenia."
- Eskalacja: "Zapytaj mnie, gdy odpowiedź zależy od tego, czy dotyczy produktu czy narzędzi dla developerów."

Nie wpisuj kroków typu "najpierw poszukaj, potem przeczytaj". To jest droga robota, nie Twój kontrakt.
</details>

### Rozwiązanie przykładowe

<details>
<summary><b>👉 Kliknij dopiero po własnej próbie</b></summary>

```python
my_contract = {
    "decyzja": "Czy warto zrobić spike AI code review w ciągu 90 dni?",
    "dowód": "Nie rekomenduj działania bez źródła pierwotnego i kontrargumentu.",
    "eskalacja": "Zapytaj mnie, gdy kontekst naszego zespołu zmieni zakres pilota.",
}

for name, value in my_contract.items():
    print(f"{name}: {value}")
```

To jest kontrakt, bo określa cel i granice. Nie mówi robotowi, w jakiej kolejności ma czytać Hacker News.
</details>

In [ ]:
## YOUR CODE HERE

<div class="alert alert-block alert-danger">

<h2>✅ Zaliczenie Dnia 1</h2>

<h4>Co masz teraz zrobić? (3 kroki)</h4>

<ol>
  <li>🧠 <b>Uzupełnij kontrakt</b> w Zadaniu 1.1. Wystarczą trzy konkretne zdania: decyzja, standard dowodu, moment pytania do człowieka.</li>
  <li>📸 <b>Zrób screenshot</b> swojego kontraktu i poprawnej odpowiedzi z self-checku.</li>
  <li>💬 <b>Wrzuć go na Slacka:</b> na kanał <code><a href="https://dataworkshopclub.slack.com/archives/C0AD3JJNLN9">#dwthon_day1_done</a></code> wraz z jednym zdaniem: "Mój robot może zmieniać plan, ale nie może zmienić ...".</li>
</ol>
<p>🆘 <b>Brak dostępu do Slacka?</b> Wyślij screen lub link na <code>hello@dataworkshop.eu</code>.</p>

To nie jest sprawdzian z teorii. Chcę zobaczyć, czy potrafisz określić granicę robota zanim damy mu więcej autonomii. Jutro wrócimy do tego samego kontraktu i zobaczysz, jak naprawdę steruje decyzjami silnika.

</div>


## 🎯 Zadanie bonusowe: "Biorę udział"

Zrób też jedną rzecz dla siebie. Dzień 3 jest momentem, w którym ten sam robot dostanie interfejs i zacznie odpowiadać na Twoje założenia. Publiczna deklaracja jest prostym sposobem, żeby nie odłożyć tego momentu na później.

**Co masz zrobić w 3 minuty:**

1. Pobierz [grafikę #1](https://storage.googleapis.com/dw-cert-public/events/dwthon_planning/DW-cat-03.png) albo [grafikę #2](https://storage.googleapis.com/dw-cert-public/events/dwthon_planning/DW-color-02.png).
2. Napisz na LinkedIn: "Biorę udział w DWthon Adaptive Planning" i dodaj jedno własne zdanie z dzisiejszego dnia.
3. Oznacz @DataWorkshop i @Vladimir Alekseichenko, a potem wyślij swój link na hello@dataworkshop.eu

Najlepiej działa własna obserwacja, nie streszczenie notebooka.

Możesz zacząć tak:

> "Dziś przypomniałem sobie, że plan agenta nie jest listą kroków. Jest hipotezą, którą trzeba umieć zmienić po wyniku narzędzia."

albo tak:

> "Autonomia nie oznacza oddania wszystkiego modelowi. Oznacza, że model wybiera drogę, a system nie pozwala mu wyjść poza granice."

Oznacz @DataWorkshop i @Vladimir Alekseichenko. Jeśli chcesz odebrać bonus za dzień 1, wyślij link lub screen na `hello@dataworkshop.eu`.

## Co zabierasz z dnia 1

Dużo konceptów, mało kodu. Tak miało być. Pierwszy dzień jest fundamentem. Bez niego jutrzejsze klasy Pydantic byłyby tylko klepaniem pól, a nie projektem robota, któremu można delegować pracę.

1. **Software 3.0 to decyzje w języku, nie magia modelu.** LLM jest potrzebny tam, gdzie kilka poprawnych faktów trzeba zinterpretować razem.
2. **Poziom 5 przypomina samodzielnego analityka.** Dostaje cel, kryteria i budżet. Sam wybiera drogę, ale nie zmienia celu.
3. **Plan jest hipotezą, nie checklistą.** Pierwszy wynik może dodać nowe pytanie albo unieważnić stare.
4. **Kontrakt i plan to dwie różne rzeczy.** Człowiek zmienia kontrakt jawnie. Model poprawia plan przez `PlanPatch`.
5. **`NextStep` jest jedną decyzją po obserwacji.** Zapisuje sens wyniku, priorytet, zmianę planu i jedną kolejną akcję.
6. **Schema-Guided Reasoning daje kontrolowaną autonomię.** Model wybiera z dostępnych dróg, a kod może usunąć drogę, która przestała być rozsądna.
7. **Transparentność jest częścią autonomii.** Każda decyzja ma powód, a później można zobaczyć drogę robota, nie tylko jego końcowy raport.

Jutro otworzymy ten system. Kod zobaczy kilka poprawnych faktów. Tylko model zrozumie, że razem zmieniają kierunek badania. Mniej literek, więcej kodu. Przygotuj kawę. ☕
